In [153]:
import json
import polars as pl

def load_data(file_path):
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data

def create_dataframe():

    data = load_data('/Users/jameslee/cj/cj_challenge/dataset/Data_Set.json')
        
    destinations = data['destinations']
    df_destinations = pl.DataFrame({
        'destination_id': [d['destination_id'] for d in destinations],
        'longitude': [d['location']['longitude'] for d in destinations],
        'latitude': [d['location']['latitude'] for d in destinations]
    })
        
    orders = data['orders']
    df_orders = pl.DataFrame({
        'order_number': [o['order_number'] for o in orders],
        'box_id': [o['box_id'] for o in orders],
        'destination': [o['destination'] for o in orders],
        'width': [o['dimension']['width'] for o in orders],
        'length': [o['dimension']['length'] for o in orders],
        'height': [o['dimension']['height'] for o in orders]
    })

    depot = data['depot']
    depot_row = pl.DataFrame({
    'Vehicle_ID': pl.Series([0], dtype=pl.Int32),
    'Route_Order': pl.Series([0], dtype=pl.Int32),
    'Destination': 'Depot',
    'Order_Number': pl.Series([None], dtype=pl.Int64),
    'Box_ID': pl.Series([None], dtype=pl.String),
    'Stacking_Order': pl.Series([None], dtype=pl.Int32),
    'Lower_Left_X': pl.Series([None], dtype=pl.Int32),
    'Lower_Left_Y': pl.Series([None], dtype=pl.Int32),
    'Lower_Left_Z': pl.Series([None], dtype=pl.Int32),
    'Longitude': depot['location']['longitude'], 
    'Latitude': depot['location']['latitude']
    })

    depot_row = depot_row.with_columns([
    pl.lit(depot['dimension']['width']).cast(pl.Int64).alias('Box_Width'),
    pl.lit(depot['dimension']['length']).cast(pl.Int64).alias('Box_Length'),
    pl.lit(depot['dimension']['height']).cast(pl.Int64).alias('Box_Height')
])

    result = df_orders.join(
            df_destinations,
            left_on='destination',
            right_on='destination_id',
            how='left'
        )
    result = result.with_columns([
        pl.lit(0).alias('Vehicle_ID'),
        pl.lit(0).alias('Route_Order'),
        pl.lit(0).alias('Stacking_Order'),
        pl.lit(0).alias('Lower_Left_X'),
        pl.lit(0).alias('Lower_Left_Y'),
        pl.lit(0).alias('Lower_Left_Z'),
    ])

    result = result.select([
            'Vehicle_ID',
            'Route_Order',
            pl.col('destination').alias('Destination'),
            pl.col('order_number').alias('Order_Number'),
            pl.col('box_id').alias('Box_ID'),
            'Stacking_Order',
            'Lower_Left_X',
            'Lower_Left_Y',
            'Lower_Left_Z',
            pl.col('longitude').alias('Longitude'),
            pl.col('latitude').alias('Latitude'),
            pl.col('width').alias('Box_Width'),
            pl.col('length').alias('Box_Length'),
            pl.col('height').alias('Box_Height'),
        ])
        
    result = pl.concat([depot_row, result])
    result = result.with_columns(
        (pl.col('Box_Width') * pl.col('Box_Length') * pl.col('Box_Height')).alias('Volume')
    )
    return result

df = create_dataframe()
df.write_csv('/Users/jameslee/cj/cj_challenge/dataset/dataset_df.csv')

In [154]:
matrix = pl.read_csv(
    "/Users/jameslee/cj/cj_challenge/dataset/distance-data.txt", 
    separator="\t", 
    has_header=True, # first row is header (column names)
)
matrix.write_csv("/Users/jameslee/cj/cj_challenge/dataset/matrix_df.csv")